In [1]:
import pandas as pd

# Load data
movies = pd.read_csv("title.basics.tsv", sep="\t", low_memory=False)

# Filter only movies
movies = movies[movies['titleType'] == 'movie']

# Remove missing years
movies = movies[movies['startYear'] != '\\N']

movies = movies[['tconst', 'primaryTitle', 'startYear', 'genres']]

movies.columns = ['imdb_id', 'title', 'year', 'genres']

print(movies.shape)
print(movies.head())

(630818, 4)
       imdb_id                          title  year  \
8    tt0000009                     Miss Jerry  1894   
144  tt0000147  The Corbett-Fitzsimmons Fight  1897   
498  tt0000502                       Bohemios  1905   
570  tt0000574    The Story of the Kelly Gang  1906   
587  tt0000591               The Prodigal Son  1907   

                         genres  
8                       Romance  
144      Documentary,News,Sport  
498                          \N  
570  Action,Adventure,Biography  
587                       Drama  


In [2]:
ratings = pd.read_csv("title.ratings.tsv", sep="\t")

movies = movies.merge(ratings, left_on="imdb_id", right_on="tconst", how="left")

movies = movies[['imdb_id', 'title', 'year', 'genres', 'averageRating', 'numVotes']]
movies.columns = ['imdb_id', 'title', 'year', 'genres', 'rating', 'votes']

movies = movies[movies['votes'] >= 1000]

movies['year'] = pd.to_numeric(movies['year'])
movies = movies[movies["year"] >= 1950]

print(movies.shape)
print(movies.head())

(45867, 6)
         imdb_id                           title  year  \
24357  tt0035423                  Kate & Leopold  2001   
26997  tt0038687              Let There Be Light  1980   
28473  tt0040559  The Machine to Kill Bad People  1952   
28898  tt0041181                      Black Hand  1950   
29068  tt0041387                         Francis  1950   

                         genres  rating    votes  
24357    Comedy,Fantasy,Romance     6.4  93742.0  
26997           Documentary,War     7.4   2168.0  
28473            Comedy,Fantasy     6.7   1116.0  
28898  Crime,Film-Noir,Thriller     6.4   1053.0  
29068       Comedy,Drama,Family     6.4   1691.0  


In [4]:
movie_ids = set(movies["imdb_id"])

chunks = pd.read_csv(
    "title.akas.tsv",
    sep="\t",
    chunksize=100000
)
filtered_chunks = []

for chunk in chunks:
    # Filter only Indian + matching movies
    chunk = chunk[(chunk["region"] == "IN") & (chunk["language"] == "hi") & (chunk["titleId"].isin(movie_ids))]

    filtered_chunks.append(chunk)

akas_india = pd.concat(filtered_chunks)

# Remove duplicates
akas_india = akas_india.drop_duplicates(subset="titleId")
akas_india = akas_india[["titleId", "region", "language"]]

# Merge safely
movies = movies.merge(
    akas_india,
    left_on="imdb_id",
    right_on="titleId",
    how="inner"
)

movies = movies.drop(columns=["titleId"])
movies = movies.reset_index(drop=True)
print(movies.shape)
print(movies.head())

(13226, 8)
     imdb_id                  title  year                    genres  rating  \
0  tt0042192          All About Eve  1950                     Drama     8.2   
1  tt0042276         Born Yesterday  1950      Comedy,Drama,Romance     7.5   
2  tt0042332             Cinderella  1950  Animation,Family,Fantasy     7.3   
3  tt0042648  Kiss Tomorrow Goodbye  1950  Crime,Film-Noir,Thriller     7.1   
4  tt0042876               Rashomon  1950       Crime,Drama,Mystery     8.1   

      votes region language  
0  148770.0     IN       hi  
1   13562.0     IN       hi  
2  186463.0     IN       hi  
3    2840.0     IN       hi  
4  195837.0     IN       hi  


In [5]:
# crew(directors)
crew_chunks = pd.read_csv(
    "title.crew.tsv",
    sep="\t",
    usecols=["tconst", "directors"],
    chunksize=100000
)

crew_list = []

for chunk in crew_chunks:
    chunk = chunk[chunk["tconst"].isin(movie_ids)]
    crew_list.append(chunk)

crew = pd.concat(crew_list).drop_duplicates("tconst")

# Split directors
crew["directors"] = crew["directors"].str.split(",")

# principals(actors)
principals_chunks = pd.read_csv(
    "title.principals.tsv",
    sep="\t",
    usecols=["tconst", "nconst", "category", "ordering"],
    chunksize=100000
)

principals_list = []

for chunk in principals_chunks:
    chunk = chunk[chunk["tconst"].isin(movie_ids)]
    chunk = chunk[chunk["category"].isin(["actor", "actress"])]
    principals_list.append(chunk)

principals = pd.concat(principals_list)

# Keep top 5 actors per movie (important)
principals = principals.sort_values("ordering").groupby("tconst").head(5)

actor_ids = set(principals["nconst"])
director_ids = set(x for sublist in crew["directors"].dropna() for x in sublist)

all_ids = actor_ids.union(director_ids)

name_chunks = pd.read_csv(
    "name.basics.tsv",
    sep="\t",
    usecols=["nconst", "primaryName"],
    chunksize=100000
)

name_list = []

for chunk in name_chunks:
    chunk = chunk[chunk["nconst"].isin(all_ids)]
    name_list.append(chunk)

names = pd.concat(name_list)

# Fast lookup
name_dict = dict(zip(names["nconst"], names["primaryName"]))

principals["actor_name"] = principals["nconst"].map(name_dict)

actors_grouped = (
    principals.groupby("tconst")["actor_name"]
    .apply(lambda x: ", ".join(x.dropna()))
    .reset_index()
)

actors_grouped.rename(columns={"actor_name": "actors"}, inplace=True)

crew_exploded = crew.explode("directors")
crew_exploded["director_name"] = crew_exploded["directors"].map(name_dict)

directors_grouped = (
    crew_exploded.groupby("tconst")["director_name"]
    .apply(lambda x: ", ".join(x.dropna()))
    .reset_index()
)

directors_grouped.rename(columns={"director_name": "directors"}, inplace=True)

# merge into movies
movies = movies.merge(
    actors_grouped,
    left_on="imdb_id",
    right_on="tconst",
    how="left"
)

movies = movies.merge(
    directors_grouped,
    left_on="imdb_id",
    right_on="tconst",
    how="left"
)

movies = movies[[
    "imdb_id",
    "title",
    "year",
    "genres",
    "rating",
    "votes",
    "actors",
    "directors"
]]

print(movies.shape)
print(movies.head())

(13226, 8)
     imdb_id                  title  year                    genres  rating  \
0  tt0042192          All About Eve  1950                     Drama     8.2   
1  tt0042276         Born Yesterday  1950      Comedy,Drama,Romance     7.5   
2  tt0042332             Cinderella  1950  Animation,Family,Fantasy     7.3   
3  tt0042648  Kiss Tomorrow Goodbye  1950  Crime,Film-Noir,Thriller     7.1   
4  tt0042876               Rashomon  1950       Crime,Drama,Mystery     8.1   

      votes                                             actors  \
0  148770.0  Bette Davis, Anne Baxter, George Sanders, Cele...   
1   13562.0  Judy Holliday, William Holden, Broderick Crawf...   
2  186463.0  Ilene Woods, James MacDonald, James MacDonald,...   
3    2840.0  James Cagney, Barbara Payton, Helena Carter, W...   
4  195837.0  Toshirô Mifune, Machiko Kyô, Masayuki Mori, Ta...   

                                         directors  
0                             Joseph L. Mankiewicz  
1          

In [ ]:
# Clean genres
movies["genres"] = movies["genres"].str.replace(",", ", ", regex=False)

def clean_text(x):
    return str(x).lower().replace(",", "")

movies["flt_genre"] = movies["genres"].apply(clean_text)
movies["flt_director"] = movies["directors"].apply(clean_text)
movies["flt_actors"] = movies["actors"].apply(clean_text)

def create_tags(row):
    return (
        (row["flt_genre"] + " ") * 3 +
        (row["flt_director"] + " ") * 2 +
        (row["flt_actors"])
    )

movies["tags"] = movies.apply(create_tags, axis=1)

C = movies['rating'].mean()
m = movies['votes'].quantile(0.75)

def weighted_rating(x):
    v = x['votes']
    R = x['rating']
    return (v/(v+m) * R) + (m/(v+m) * C)

movies['score'] = movies.apply(weighted_rating, axis=1)

movies.drop(columns=["flt_genre", "flt_director", "flt_actors"], inplace=True)
movies.to_csv("movies.csv", index=False)